In [1]:
import pandas as pd 
import numpy as np 
import matplotlib.pyplot as plt

In [2]:
fireball = pd.read_csv('fireballs.csv',skiprows=[1])
#fireball
fireball = fireball.rename(columns={
    "Velocity Components (km/s)": "Velocity (X-component)(km/s)",
    "Unnamed: 5": "Velocity (Y-component)(km/s)",
    "Unnamed: 6": "Velocity (Z-component)(km/s)"
})

In [ ]:
# Basic structure of the dataset
print("Shape:", fireball.shape) # (rows, columns)
print("\nColumn names:") #column names
print(fireball.columns.tolist())

print("\nData types:")
print(fireball.dtypes)

In [ ]:
# Example observations
fireball.head()

In [ ]:
# Missing values in every column
missing_all = pd.DataFrame({
    "missing_count": fireball.isna().sum(),
    "missing_percentage": fireball.isna().mean() * 100
})

missing_all

In [6]:

total_observations = len(fireball)

altitude_available = fireball["Altitude (km)"].notna().sum()

complete_velocity = (
    fireball[
        [
            "Velocity (X-component)(km/s)",
            "Velocity (Y-component)(km/s)",
            "Velocity (Z-component)(km/s)"
        ]
    ]
    .notna()
    .all(axis=1)
    .sum()
)

altitude_and_velocity = (
    fireball[
        [
            "Altitude (km)",
            "Velocity (X-component)(km/s)",
            "Velocity (Y-component)(km/s)",
            "Velocity (Z-component)(km/s)"
        ]
    ]
    .notna()
    .all(axis=1)
    .sum()
)

print("Total observations:", total_observations)
print("Observations with altitude:", altitude_available)
print("Observations with complete velocity:", complete_velocity)
print(
    "Observations with altitude and complete velocity:",
    altitude_and_velocity
)

Total observations: 1073
Observations with altitude: 601
Observations with complete velocity: 361
Observations with altitude and complete velocity: 358


 Initial Dataset Inspection

The dataset contains 1,073 observations and 9 variables. The initial inspection included the dataset dimensions, column names, data types, example records, and missing values. Missing data is concentrated mainly in the location, altitude, and velocity measurements. Latitude and longitude are each missing for 186 observations (17.33%), altitude is missing for 472 observations (43.99%), and the three velocity components each contain 712 missing values (66.36%). Of the 1,073 total observations, 601 have altitude measurements, 361 have complete velocity data for all three velocity components, and 358 contain both altitude and complete velocity data. Missing observations were retained rather than immediately removing incomplete rows, since the availability of altitude, location, and velocity measurements varies between events and different stages of the analysis may require different subsets of the dataset.

### Data cleaning
For the purposes of the machine learning model, there seems to be little value for latitude, longitude and peak brightness. It's better to begin without them.  But, for the sake of it, let's convert the values here. Use a copy of the original dataframe, helps if we later need to go back to it.

In [7]:
# latitude and longitude conversion helper function
def parse_coord(s):
    # Remove whitespace and get the cardinal direction letter if present
    s = s.str.strip()
    direction = s.str.extract(r'([NSEWnsew])$', expand=False)

    # Convert the numeric part to float
    val = pd.to_numeric(s.str.replace(r'[NSEWnsew]$', '', regex=True))

    # Multiply by -1 for South and West
    return val.where(~direction.str.upper().isin(['S', 'W']), -val)

In [ ]:
fireball_converted = fireball.copy()
fireball_converted["Peak Brightness Date/Time (UT)"] = pd.to_datetime(fireball["Peak Brightness Date/Time (UT)"], format='%Y-%m-%d %H:%M:%S')
fireball_latitude = parse_coord(fireball["Latitude (deg.)"])
fireball_longitude = parse_coord(fireball["Longitude (deg.)"])
fireball_converted["Latitude (deg.)"] = fireball_latitude
fireball_converted["Longitude (deg.)"] = fireball_longitude

fireball_converted
# fireball

In [ ]:
# These can be used for the main models

fireball_converted["Velocity Magnitude (km/s)"] = np.sqrt(
    fireball_converted["Velocity (X-component)(km/s)"]**2
    + fireball_converted["Velocity (Y-component)(km/s)"]**2
    + fireball_converted["Velocity (Z-component)(km/s)"]**2
)
fireball_converted["Log10 Impact Energy (kt)"] = np.log10(
    fireball_converted["Calculated Total Impact Energy (kt)"]
)


fireball_ml = fireball_converted[
    [
        "Altitude (km)",
        "Velocity (X-component)(km/s)",
        "Velocity (Y-component)(km/s)",
        "Velocity (Z-component)(km/s)",
        "Velocity Magnitude (km/s)",
        "Calculated Total Impact Energy (kt)",
        "Log10 Impact Energy (kt)"
    ]
]

fireball_ml_simple = fireball_ml.dropna()

fireball_ml_simple

### Notes on imputation
- There's a good chance the data is "Missing Not At Random", meaning the fireballs were too weak to track properly
- If imputing, use Multivariate Imputation by Chained Equations (MICE) (scikit IterativeImputer) combined with Missingness Indicator Flags (was data present or was it imputated).

In [ ]:
print("Impact Energy Summary:")
display(
    fireball_ml_simple[
        ["Calculated Total Impact Energy (kt)", "Log10 Impact Energy (kt)"]
    ].describe()
)
plt.figure(figsize=(8, 5))

plt.hist(
    fireball_ml_simple["Calculated Total Impact Energy (kt)"],
    bins=40,
    range=(0, 5),
    edgecolor="black"
)

plt.xlabel("Impact Energy (kt)")
plt.ylabel("Number of Fireballs")
plt.title("Distribution of Impact Energy")

plt.show()
fireball_ml_simple["Calculated Total Impact Energy (kt)"].sort_values(ascending=False).head(10)
above_5 = (
    fireball_ml_simple["Calculated Total Impact Energy (kt)"] > 5
).sum()

print(
    f"Note: {above_5} observations above 5 kt are not shown in this histogram. "
    "The plot is restricted to 0 - 5 kt so that the distribution of the majority "
    "of fireballs can be examined clearly. The higher-energy observations remain "
    "in the dataset and are not removed from the analysis."
)
high_energy_fireballs = fireball_ml_simple[
    fireball_ml_simple["Calculated Total Impact Energy (kt)"] > 5
].sort_values(
    by="Calculated Total Impact Energy (kt)",
    ascending=False
)

display(high_energy_fireballs)

plt.figure(figsize=(8, 5))

plt.hist(
    fireball_ml_simple["Log10 Impact Energy (kt)"],
    bins=40,
    edgecolor="black"
)

plt.xlabel("Log10 Impact Energy")
plt.ylabel("Number of Fireballs")
plt.title("Distribution of Log10 Impact Energy")

plt.show()

Now I shall make three scatter plots: altitude vs log impact energy, velocity magnitude vs log impact energy, and altitude vs velocity magnitude. I shall use  log impact energy rather than raw impact energy here because we  already showed that raw impact energy is extremely right-skewed and dominated by a handful of extreme events.

In [ ]:
plt.figure(figsize=(8, 5))

plt.scatter(
    fireball_ml_simple["Altitude (km)"],
    fireball_ml_simple["Log10 Impact Energy (kt)"],
    alpha=0.6
)

plt.xlabel("Altitude at Peak Brightness (km)")
plt.ylabel("Log10 Impact Energy")
plt.title("Altitude vs Log10 Impact Energy")

plt.grid(alpha=0.3)
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))

plt.scatter(
    fireball_ml_simple["Velocity Magnitude (km/s)"],
    fireball_ml_simple["Log10 Impact Energy (kt)"],
    alpha=0.6
)

plt.xlabel("Velocity Magnitude (km/s)")
plt.ylabel("Log10 Impact Energy")
plt.title("Velocity Magnitude vs Log10 Impact Energy")

plt.grid(alpha=0.3)
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))

plt.scatter(
    fireball_ml_simple["Altitude (km)"],
    fireball_ml_simple["Velocity Magnitude (km/s)"],
    alpha=0.6
)

plt.xlabel("Altitude at Peak Brightness (km)")
plt.ylabel("Velocity Magnitude (km/s)")
plt.title("Altitude vs Velocity Magnitude")

plt.grid(alpha=0.3)
plt.show()

In [14]:
pairs = [
    ("Altitude (km)", "Log10 Impact Energy (kt)"),
    ("Velocity Magnitude (km/s)", "Log10 Impact Energy (kt)"),
    ("Altitude (km)", "Velocity Magnitude (km/s)")
]

for x, y in pairs:
    pearson = fireball_ml_simple[x].corr(
        fireball_ml_simple[y],
        method="pearson"
    )

    spearman = fireball_ml_simple[x].corr(
        fireball_ml_simple[y],
        method="spearman"
    )

    print(f"{x} vs {y}")
    print(f"  Pearson:  {pearson:.3f}")
    print(f"  Spearman: {spearman:.3f}")
    print()

Altitude (km) vs Log10 Impact Energy (kt)
  Pearson:  -0.142
  Spearman: -0.078

Velocity Magnitude (km/s) vs Log10 Impact Energy (kt)
  Pearson:  -0.009
  Spearman: 0.025

Altitude (km) vs Velocity Magnitude (km/s)
  Pearson:  0.392
  Spearman: 0.144



In [15]:
pairs_raw = [
    ("Altitude (km)", "Calculated Total Impact Energy (kt)"),
    ("Velocity Magnitude (km/s)", "Calculated Total Impact Energy (kt)")
]

for x, y in pairs_raw:
    pearson = fireball_ml_simple[x].corr(
        fireball_ml_simple[y],
        method="pearson"
    )

    spearman = fireball_ml_simple[x].corr(
        fireball_ml_simple[y],
        method="spearman"
    )

    print(f"{x} vs {y}")
    print(f"  Pearson:  {pearson:.3f}")
    print(f"  Spearman: {spearman:.3f}")
    print()

Altitude (km) vs Calculated Total Impact Energy (kt)
  Pearson:  -0.081
  Spearman: -0.078

Velocity Magnitude (km/s) vs Calculated Total Impact Energy (kt)
  Pearson:  0.010
  Spearman: 0.025

